# 02 Data Cleaning

Clean and transform the Olist datasets, create analysis fields and save processed data.


In [584]:
from pathlib import Path
import pandas as pd
import numpy as np


In [585]:
project_root = Path.cwd().parent

raw_data_path = project_root / "data" / "raw" / "Dataset"
processed_data_path = project_root / "data" / "processed"

processed_data_path.mkdir(parents = True, exist_ok = True)

print(raw_data_path)
print(processed_data_path)

/Users/xsha/Desktop/customer-product-analysis/data/raw/Dataset
/Users/xsha/Desktop/customer-product-analysis/data/processed


In [586]:
# Read all the csv files

customers = pd.read_csv(raw_data_path / "olist_customers_dataset.csv")

orders = pd.read_csv(raw_data_path / "olist_orders_dataset.csv")

order_items = pd.read_csv(raw_data_path / "olist_order_items_dataset.csv")

products = pd.read_csv(raw_data_path / "olist_products_dataset.csv")

reviews = pd.read_csv(raw_data_path / "olist_order_reviews_dataset.csv")

category_translation = pd.read_csv(raw_data_path / "product_category_name_translation.csv")

In [587]:
# Check if the csv files can be read successfully

dataframes = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "reviews": reviews,
    "category_translation": category_translation
}

for name, df in dataframes.items():
    print(name, df.shape)

# (rows, columns)

customers (99441, 5)
orders (99441, 8)
order_items (112650, 7)
products (32951, 9)
reviews (99224, 7)
category_translation (71, 2)


In [588]:
# Use the copys of original datasets for data cleaning

customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
products_clean = products.copy()
reviews_clean = reviews.copy()
category_translation_clean = category_translation.copy()

In [589]:
# Clean the data formats

clean_dataframes = [
    customers_clean,
    orders_clean,
    order_items_clean,
    products_clean,
    reviews_clean,
    category_translation_clean
]

for df in clean_dataframes:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

In [590]:
# Check the duplicates

for name, df in {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "products": products_clean,
    "reviews": reviews_clean,
    "category_translation": category_translation_clean
}.items():
    print(name, df.duplicated().sum())

customers 0
orders 0
order_items 0
products 0
reviews 0
category_translation 0


In [591]:
# Delete the duplicates

customers_clean = customers_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
reviews_clean = reviews_clean.drop_duplicates()
category_translation_clean = category_translation_clean.drop_duplicates()

In [592]:
# Check if the major IDs in each sheets are unique

print("Unique customer_id:", customers_clean["customer_id"].is_unique)
print("Unique order_id:", orders_clean["order_id"].is_unique)
print("Unique product_id:", products_clean["product_id"].is_unique)


# Check if there are any duplicated items

duplicate_order_items = order_items_clean.duplicated(subset = ["order_id", "order_item_id"]).sum()
print("Duplicate order items:", duplicate_order_items)

Unique customer_id: True
Unique order_id: True
Unique product_id: True
Duplicate order items: 0


In [593]:
orders_clean.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [594]:
# Convert text to datetime

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for columns in order_date_columns:
    orders_clean[columns] = pd.to_datetime(orders_clean[columns], errors = 'coerce')


# Clean order_items

order_items_clean["shipping_limit_date"] = pd.to_datetime(order_items_clean["shipping_limit_date"], errors = 'coerce')


# Clean reviews

review_date_columns = ["review_creation_date", "review_answer_timestamp"]

for columns in review_date_columns:
    reviews_clean[columns] = pd.to_datetime(reviews_clean[columns], errors = 'coerce')

In [595]:
# Add purchase year and month in orders_clean

orders_clean["purchase_year"] = (orders_clean["order_purchase_timestamp"].dt.year)

orders_clean["purchase_month"] = (orders_clean["order_purchase_timestamp"].dt.month)

orders_clean["purchase_year_month"] = (orders_clean["order_purchase_timestamp"].dt.to_period("M").astype(str))


# Add purchase day of week and hour

orders_clean["purchase_day_of_week"] = (orders_clean["order_purchase_timestamp"].dt.day_name())

orders_clean["purchase_hour"] = (orders_clean["order_purchase_timestamp"].dt.hour)

In [596]:
# Calculate the actual delivery days

orders_clean["delivery_days"] = (orders_clean["order_delivered_customer_date"] - orders_clean["order_purchase_timestamp"]).dt.days


# Calculate the differences between acutal delivery days and estimated delivery days

orders_clean["delivery_delay_days"] = (orders_clean["order_delivered_customer_date"] - orders_clean["order_estimated_delivery_date"]).dt.days


# If it is delayed

orders_clean["is_delayed"] = np.where(orders_clean["delivery_delay_days"] > 0, 1, 0)


# For the missing delivery

missing_delivery_mash = (orders_clean["order_delivered_customer_date"].isna())

orders_clean.loc[missing_delivery_mash, "is_delayed"] = np.nan

In [597]:
# Check the results

orders_clean[[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "delivery_delay_days",
        "is_delayed"
    ]].head()

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,is_delayed
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,0.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,0.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,0.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,0.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,0.0


## Product category translation

In [598]:
category_translation_clean.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [599]:
# Merge

products_clean = products_clean.merge(category_translation_clean, on = "product_category_name", how = "left")

In [600]:
# Rename the column

products_clean = products_clean.rename(columns = {"product_category_name_english": "product_category"})

In [601]:
# Missing product categories

products_clean["product_category"] = (products_clean["product_category"].fillna("unknown"))

In [602]:
# Check the results

products_clean[[
    "product_id",
    "product_category_name",
    "product_category"
]].head()

,product_id,product_category_name,product_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


In [603]:
# Check the percentage of translations that were unsuccessful

unknown_category_count = (products_clean["product_category"] == "unknown").sum()

print("Unknown categories: ", unknown_category_count)

Unknown categories:  623
